In [20]:
import sys
sys.path.append('../src/features')

In [23]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

In [21]:
from transformers import *
from preprocessing import *
from sklearn.base import BaseEstimator, TransformerMixin

In [4]:
path = '/Users/qianj/Library/Mobile Documents/com~apple~CloudDocs/AA_2024-2026 job/Erdos_Project/original_dataset'
file = 'O_PROV'

In [14]:
# build transformer
class MedicareDFTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,percentage = 0.005, 
                 use_label_data=False,label_types=None, label_col1='Tot_Risk',label_col2='Tot_Mdcr_Pymt_Amt',
                 curr_cols=None,
                 copy=True):
        self.percentage = percentage
        self.use_label_data = use_label_data
        self.label_types = label_types
        self.label_col1 = label_col1
        self.label_col2 = label_col2
        self.curr_cols = curr_cols or ["Log_Tot_Mdcr_Pymt_Amt", "Tot_Mdcr_Pymt_Amt", "year"]
        self.copy = copy
    
    def fit(self,X,y=None):
        return self
    
    def transform(self,X):
        df = X.copy() if self.copy else X
        df = clean_up_type(df)
        df = umbrella_types(df)
        df = umbrella_states(df)
        df = check_non_integer_columns(self.percentage, df)
        df = total_risk(df)

        if self.use_label_data:
            df = label_data(df, self.label_types, self.label_col1, self.label_col2)

        df = log_columns(df)
        df = push_one_year(df, curr_cols=self.curr_cols)
        return df

In [24]:
# load raw data
df = combine_keep_cols_add_year_label_df(path,file)
df

,Rndrng_NPI,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,Rndrng_Prvdr_Mdcr_Prtcptg_Ind,Tot_HCPCS_Cds,Tot_Benes,Tot_Srvcs,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,Tot_Mdcr_Pymt_Amt,Bene_Avg_Age,Bene_Avg_Risk_Scre,year
0,1003000126,I,MD,Internal Medicine,Y,22,665,1648.0,395335.00,146521.84,116332.66,74,2.1114,2013
1,1003000134,I,IL,Pathology,Y,13,3939,7517.0,1211425.00,282079.49,217960.62,76,1.0156,2013
2,1003000142,I,OH,Anesthesiology,Y,42,144,661.0,197224.00,63334.30,49752.77,63,1.5662,2013
3,1003000407,I,PA,Family Practice,Y,37,436,1683.0,240818.00,174786.95,138741.21,76,1.8967,2013
4,1003000423,I,OH,Obstetrics/Gynecology,Y,33,63,320.0,31637.00,13176.47,10320.43,56,1.1882,2013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12232189,1992999569,I,CA,Optometry,Y,6,16,24.0,3380.00,3010.25,1588.78,72,0.8738,2023
12232190,1992999775,O,OR,Ambulatory Surgical Center,Y,24,228,317.0,2682450.00,1076800.05,856134.21,74,0.9921,2023
12232191,1992999817,I,CA,Orthopedic Surgery,Y,13,18,44.0,21574.18,7201.29,5600.89,71,0.7118,2023
12232192,1992999825,I,WA,Otolaryngology,Y,34,405,646.0,235765.00,91260.38,69628.19,77,1.0682,2023


In [25]:
df = MedicareDFTransformer(
                    use_label_data=True,
                    label_types=['APP','PrimaryCare','MedicalSpecialtyOther','LabPathology','PharmacyNutrition']
                    ).fit_transform(df)
df

,Rndrng_NPI,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,Rndrng_Prvdr_Mdcr_Prtcptg_Ind,Tot_HCPCS_Cds,Tot_Benes,Tot_Srvcs,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,...,Log_Tot_Mdcr_Alowd_Amt,Log_Tot_Risk,Log_APP_Tot_Risk,Log_PrimaryCare_Tot_Risk,Log_MedicalSpecialtyOther_Tot_Risk,Log_LabPathology_Tot_Risk,Log_PharmacyNutrition_Tot_Risk,Log_Tot_Mdcr_Pymt_Amt,Tot_Mdcr_Pymt_Amt,current_year
0,1003000126,I,MD,PrimaryCare,Y,22,665,1648.0,395335.0,146521.84,...,11.894930,7.247138,0.693147,0.000000,0.693147,1.098612,0.693147,12.175990,194073.09,2014
1,1003000134,I,IL,LabPathology,Y,13,3939,7517.0,1211425.0,282079.49,...,12.549944,8.294162,0.693147,0.693147,0.693147,0.693147,0.693147,12.410118,245270.71,2014
2,1003000142,I,OH,Anesthesia,Y,42,144,661.0,197224.0,63334.30,...,11.056182,5.418466,0.693147,0.693147,0.693147,1.098612,0.693147,10.855532,51820.04,2014
3,1003000407,I,PA,PrimaryCare,Y,37,436,1683.0,240818.0,174786.95,...,12.071323,6.717758,0.693147,0.000000,0.693147,1.098612,0.693147,11.360222,85838.44,2014
4,1003000423,I,OH,OBGYN,Y,33,63,320.0,31637.0,13176.47,...,9.486188,4.315574,0.693147,0.693147,0.693147,1.098612,0.693147,9.158164,9491.61,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10196593,1992999551,I,CA,PrimaryCare,Y,26,230,1103.0,145838.0,86140.00,...,11.363729,5.476627,0.693147,0.000000,0.693147,1.098612,0.693147,11.143836,69136.34,2023
10196594,1992999569,I,CA,VisionHearing,Y,4,15,25.0,3660.0,3159.02,...,8.058017,2.439779,0.693147,0.693147,0.693147,1.098612,0.693147,7.370722,1588.78,2023
10196595,1992999775,O,OR,FacilitySupplierProgram,Y,21,208,323.0,2774700.0,1209276.66,...,14.005533,5.307285,0.693147,0.693147,0.693147,1.098612,0.693147,13.660182,856134.21,2023
10196596,1992999825,I,WA,SurgeryOther,Y,35,341,570.0,200383.3,80987.39,...,11.302049,5.913555,0.693147,0.693147,0.693147,1.098612,0.693147,11.150925,69628.19,2023


In [26]:
# preprocessing
label_cols = ['APP_Tot_Risk','PrimaryCare_Tot_Risk','MedicalSpecialtyOther_Tot_Risk','LabPathology_Tot_Risk','PharmacyNutrition_Tot_Risk']
df = label_num_to_cat(df,label_cols)
df = enc_df(df)
df

,Rndrng_NPI,Tot_HCPCS_Cds,Tot_Benes,Tot_Srvcs,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,Bene_Avg_Age,Bene_Avg_Risk_Scre,Tot_Risk,Log_Tot_HCPCS_Cds,...,APP_Tot_Risk_1,PrimaryCare_Tot_Risk_1,PrimaryCare_Tot_Risk_2,MedicalSpecialtyOther_Tot_Risk_2,MedicalSpecialtyOther_Tot_Risk_1,LabPathology_Tot_Risk_3,LabPathology_Tot_Risk_2,LabPathology_Tot_Risk_1,PharmacyNutrition_Tot_Risk_2,PharmacyNutrition_Tot_Risk_1
0,1003000126,22,665,1648.0,395335.0,146521.84,74,2.1114,1404.0810,3.091042,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1003000134,13,3939,7517.0,1211425.0,282079.49,76,1.0156,4000.4484,2.564949,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1003000142,42,144,661.0,197224.0,63334.30,63,1.5662,225.5328,3.737670,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1003000407,37,436,1683.0,240818.0,174786.95,76,1.8967,826.9612,3.610918,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1003000423,33,63,320.0,31637.0,13176.47,56,1.1882,74.8566,3.496508,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10196593,1992999551,26,230,1103.0,145838.0,86140.00,77,1.0393,239.0390,3.258097,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
10196594,1992999569,4,15,25.0,3660.0,3159.02,73,0.7647,11.4705,1.386294,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
10196595,1992999775,21,208,323.0,2774700.0,1209276.66,74,0.9702,201.8016,3.044522,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
10196596,1992999825,35,341,570.0,200383.3,80987.39,75,1.0851,370.0191,3.555348,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
